In [7]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split, RandomizedSearchCV
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_squared_error, r2_score
import pickle

# 1. Load Data
df = pd.read_csv('car.csv')

# 2. Preprocess
df['Years_of_Service'] = 2026 - df['Year']
df.drop(['Year', 'Car_Name'], axis=1, inplace=True) 
df = pd.get_dummies(df, drop_first=True)

# 3. Split
X = df.drop('Selling_Price', axis=1)
y = df['Selling_Price']
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# 4. Train
rf = RandomForestRegressor()
random_grid = {
    'n_estimators': [100, 200, 300],
    'max_features': ['sqrt', 'log2'],
    'max_depth': [5, 10, 15],
    'min_samples_split': [2, 5, 10],
    'min_samples_leaf': [1, 2, 5]
}

rf_random = RandomizedSearchCV(estimator=rf, param_distributions=random_grid, n_iter=5, cv=3, random_state=42)
rf_random.fit(X_train, y_train)

# 5. Evaluate
predictions = rf_random.predict(X_test)
print(f"Accuracy (R2): {r2_score(y_test, predictions)*100:.2f}%")

# 6. Save Model
with open('random_forest_model.pkl', 'wb') as file:
    pickle.dump(rf_random, file)
print("Model saved as 'random_forest_model.pkl'")


Accuracy (R2): 95.09%
Model saved as 'random_forest_model.pkl'


In [8]:
import ipywidgets as widgets
from IPython.display import display, clear_output
import pickle
import numpy as np

# Load the trained model
model = pickle.load(open('random_forest_model.pkl', 'rb'))

# Create Input Widgets
style = {'description_width': 'initial'}
year_input = widgets.IntText(value=2015, description='Year of Manufacture:', style=style)
price_input = widgets.FloatText(value=5.5, description='Showroom Price (Lakhs):', style=style)
kms_input = widgets.IntText(value=45000, description='Kilometers Driven:', style=style)
owner_input = widgets.Dropdown(options=[0, 1, 2, 3], value=0, description='Previous Owners:', style=style)
fuel_input = widgets.Dropdown(options=['Petrol', 'Diesel', 'CNG'], value='Petrol', description='Fuel Type:', style=style)
seller_input = widgets.Dropdown(options=['Dealer', 'Individual'], value='Dealer', description='Seller Type:', style=style)
trans_input = widgets.Dropdown(options=['Manual', 'Automatic'], value='Manual', description='Transmission:', style=style)

# Create a Button and Output Area
predict_btn = widgets.Button(description="Predict Price", button_style='success')
output = widgets.Output()

# Define what happens when the button is clicked
def on_button_clicked(b):
    with output:
        clear_output()
        
        # Format inputs for the model
        Years_of_Service = 2026 - year_input.value
        Fuel_Type_Diesel = 1 if fuel_input.value == 'Diesel' else 0
        Fuel_Type_Petrol = 1 if fuel_input.value == 'Petrol' else 0
        Seller_Type_Individual = 1 if seller_input.value == 'Individual' else 0
        Transmission_Manual = 1 if trans_input.value == 'Manual' else 0
        
        features = [np.array([
            price_input.value, kms_input.value, owner_input.value, Years_of_Service, 
            Fuel_Type_Diesel, Fuel_Type_Petrol, Seller_Type_Individual, Transmission_Manual
        ])]
        
        # Predict
        prediction = model.predict(features)
        print(f"💰 Estimated Selling Price: ₹ {prediction[0]:.2f} Lakhs")

# Link the button to the function
predict_btn.on_click(on_button_clicked)

# Display the dashboard
display(widgets.VBox([
    year_input, price_input, kms_input, owner_input, 
    fuel_input, seller_input, trans_input, predict_btn, output
]))

In [9]:
import os
os.makedirs('templates', exist_ok=True)

In [10]:
import os
os.makedirs('static', exist_ok=True)